In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nDuplicate rows:", df.duplicated().sum())

print("\nChurn counts:")
print(df["Churn"].value_counts())

print("\nChurn percentages:")
print(
    (df["Churn"].value_counts(normalize=True) * 100).round(2)
)

Column names:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Data types:
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

Duplicate rows: 0

Churn counts:
Churn
No     

In [4]:
print("TotalCharges type before conversion:", df["TotalCharges"].dtype)

blank_count = (
    df["TotalCharges"].astype(str).str.strip().eq("").sum()
)

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"].astype(str).str.strip(),
    errors="coerce"
)

print("Blank strings before conversion:", blank_count)
print("TotalCharges type after conversion:", df["TotalCharges"].dtype)
print("Missing TotalCharges after conversion:", df["TotalCharges"].isna().sum())

TotalCharges type before conversion: object
Blank strings before conversion: 11
TotalCharges type after conversion: float64
Missing TotalCharges after conversion: 11


In [5]:
# Remove the target and identifier from the predictors
X = df.drop(
    columns=["Churn", "customerID", "TenureGroup"],
    errors="ignore"
).copy()

# No = 0, Yes = 1
y = df["Churn"].map({"No": 0, "Yes": 1})

# Confirm target conversion
assert y.notna().all(), "Unexpected values found in Churn."

# Stratified 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining churn distribution:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTesting churn distribution:")
print(y_test.value_counts(normalize=True).round(3))

Training features: (5634, 19)
Testing features: (1409, 19)

Training churn distribution:
Churn
0    0.735
1    0.265
Name: proportion, dtype: float64

Testing churn distribution:
Churn
0    0.735
1    0.265
Name: proportion, dtype: float64


In [6]:
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Number of numeric features:", len(numeric_features))
print("Numeric features:")
print(numeric_features)

print("\nNumber of categorical features:", len(categorical_features))
print("Categorical features:")
print(categorical_features)

assert len(numeric_features) + len(categorical_features) == X_train.shape[1]

Number of numeric features: 4
Numeric features:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Number of categorical features: 15
Categorical features:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [7]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully.")
print(preprocessor)

Preprocessing pipeline created successfully.
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['SeniorCitizen', 'tenure', 'MonthlyCharges',
                                  'TotalCharges']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['gender', 'Partner', 'Dependents',
                                  'PhoneService', 'MultipleLines',
                                  'InternetService', 'OnlineSecurity',
   

In [8]:
logistic_default = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

print("Default Logistic Regression pipeline created.")

Default Logistic Regression pipeline created.


In [9]:
logistic_balanced = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    ))
])

print("Balanced Logistic Regression pipeline created.")

Balanced Logistic Regression pipeline created.


In [10]:
print("Training default Logistic Regression...")
logistic_default.fit(X_train, y_train)

print("\nTraining balanced Logistic Regression...")
logistic_balanced.fit(X_train, y_train)

print("\nBoth models trained successfully!")

print(
    "Default model training score:",
    round(logistic_default.score(X_train, y_train), 4)
)

print(
    "Balanced model training score:",
    round(logistic_balanced.score(X_train, y_train), 4)
)

Training default Logistic Regression...

Training balanced Logistic Regression...

Both models trained successfully!
Default model training score: 0.806
Balanced model training score: 0.7531


In [11]:
from sklearn.dummy import DummyClassifier

baseline_model = DummyClassifier(strategy="most_frequent")

baseline_model.fit(X_train, y_train)

print("Majority-class baseline trained successfully.")
print("Class predicted:", baseline_model.classes_[baseline_model.class_prior_.argmax()])

Majority-class baseline trained successfully.
Class predicted: 0


In [12]:
# Predictions
baseline_predictions = baseline_model.predict(X_test)
default_predictions = logistic_default.predict(X_test)
balanced_predictions = logistic_balanced.predict(X_test)

# Churn probabilities for ROC-AUC
baseline_probabilities = baseline_model.predict_proba(X_test)[:, 1]
default_probabilities = logistic_default.predict_proba(X_test)[:, 1]
balanced_probabilities = logistic_balanced.predict_proba(X_test)[:, 1]

print("Predictions generated for all three models.")
print("Test observations:", len(y_test))

Predictions generated for all three models.
Test observations: 1409


In [13]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

def evaluate_model(model_name, y_true, predictions, probabilities):
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "F1-Score": f1_score(
            y_true, predictions, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(y_true, probabilities)
    }

results = [
    evaluate_model(
        "Majority-Class Baseline",
        y_test,
        baseline_predictions,
        baseline_probabilities
    ),
    evaluate_model(
        "Logistic Regression (Default)",
        y_test,
        default_predictions,
        default_probabilities
    ),
    evaluate_model(
        "Logistic Regression (Balanced)",
        y_test,
        balanced_predictions,
        balanced_probabilities
    )
]

comparison = pd.DataFrame(results)

print("Evaluation metrics calculated.")

Evaluation metrics calculated.


In [14]:
comparison_display = comparison.copy()

metric_columns = [
    "Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"
]

comparison_display[metric_columns] = (
    comparison_display[metric_columns].round(4)
)

display(comparison_display)

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Majority-Class Baseline,0.7346,0.0000,0.0000,0.0000,0.5000
1,Logistic Regression (Default),0.8055,0.6572,0.5588,0.6040,0.8419
2,Logistic Regression (Balanced),0.7381,0.5043,0.7834,0.6136,0.8413


In [15]:
from sklearn.metrics import confusion_matrix

model_predictions = {
    "Majority-Class Baseline": baseline_predictions,
    "Logistic Regression (Default)": default_predictions,
    "Logistic Regression (Balanced)": balanced_predictions
}

for model_name, predictions in model_predictions.items():
    cm = confusion_matrix(y_test, predictions, labels=[0, 1])

    print(f"\n{model_name}")
    print("Rows = actual classes [No Churn, Churn]")
    print("Columns = predicted classes [No Churn, Churn]")
    print(cm)


Majority-Class Baseline
Rows = actual classes [No Churn, Churn]
Columns = predicted classes [No Churn, Churn]
[[1035    0]
 [ 374    0]]

Logistic Regression (Default)
Rows = actual classes [No Churn, Churn]
Columns = predicted classes [No Churn, Churn]
[[926 109]
 [165 209]]

Logistic Regression (Balanced)
Rows = actual classes [No Churn, Churn]
Columns = predicted classes [No Churn, Churn]
[[747 288]
 [ 81 293]]
